# KG-Path Filtering of Control-Action Tags — 03LIC_1071

Filter the control-action tags recorded per alarm episode down to only the tags that are
**knowledge-graph (KG) relevant** to the target loop `03LIC_1071`.

**Inputs**
- KG parallel groups: `DATA/LIC_1071_parallel_groups_Jun_29.xlsx` — column `DCS_Path`
- Control actions: `RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx` — sheet `control_actions`

**Steps**
1. Keep only `DCS_Path` rows that contain the `1071` tag.
2. Extract the set of **unique tags** across those filtered paths → the KG relevance list.
3. For every episode (`cluster_id`), list the tags that were operated, then split them into
   **kept** (present in the KG list) and **eliminated** (absent from the KG list).
4. Write a per-episode Excel and list every tag eliminated across all episodes.

_Note: this workbook is the 1071 parallel-groups file, so every row already contains `1071`;
the row filter is still applied so the same logic generalises to other target loops._


In [1]:
import pandas as pd
from IPython.display import display, Markdown

# ── Paths ──────────────────────────────────────────────────────────────────────
KG_PATHS_XLSX = "/home/h604827/ControlActions/DATA/LIC_1071_parallel_groups_Jun_29.xlsx"
CONTROL_ACTIONS_XLSX = (
    "/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219"
    "/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx"
)
OUTPUT_XLSX = (
    "/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219"
    "/03LIC_1071_kg_filtered_control_action_tags_per_episode.xlsx"
)
TARGET_LOOP = "1071"

# ── Step 1: build the KG relevance tag list ────────────────────────────────────
kg = pd.read_excel(KG_PATHS_XLSX, sheet_name="Sheet1")

# (1a) keep only DCS_Path rows that contain the target-loop tag
dcs_paths = kg["DCS_Path"].dropna().astype(str)
paths_with_loop = dcs_paths[dcs_paths.str.contains(TARGET_LOOP, na=False)]

# (1b) extract every unique tag across those filtered paths
kg_tags = set()
for path in paths_with_loop:
    for tag in path.split(","):
        tag = tag.strip()
        if tag:
            kg_tags.add(tag)

print(f"DCS_Path rows total       : {len(dcs_paths):,}")
print(f"Rows containing '{TARGET_LOOP}'      : {len(paths_with_loop):,}")
print(f"Unique KG-relevant tags   : {len(kg_tags):,}")
print(f"KG tags containing '{TARGET_LOOP}'   : {sorted(t for t in kg_tags if TARGET_LOOP in t)}")


DCS_Path rows total       : 853
Rows containing '1071'      : 853
Unique KG-relevant tags   : 711
KG tags containing '1071'   : ['03LCV_1071_LB', '03LIC_1071', '03LI_1071', '03LT_1071', '03LY_1071', '03ZS_1071C']


In [2]:
# ── Step 2: tags operated per episode (cluster_id) ─────────────────────────────
# Every row in the control_actions sheet is an operator control action.
# An "operated tag" for an episode = any Source tag with >=1 action in that cluster.
ca = pd.read_excel(CONTROL_ACTIONS_XLSX, sheet_name="control_actions")
ca = ca[ca["Source"].notna()].copy()

operated_per_episode = (
    ca.groupby("cluster_id")["Source"]
    .apply(lambda s: sorted(set(s)))
    .to_dict()
)

all_operated_tags = sorted(set(ca["Source"]))
print(f"Control-action rows                      : {len(ca):,}")
print(f"Episodes (clusters) with control actions : {len(operated_per_episode):,}")
print(f"Unique operated tags (all episodes)      : {len(all_operated_tags):,}")


Control-action rows                      : 42,290
Episodes (clusters) with control actions : 525
Unique operated tags (all episodes)      : 195


In [3]:
# ── Step 3: KG-filter operated tags per episode ────────────────────────────────
rows = []
for cluster_id, operated in operated_per_episode.items():
    kept = [t for t in operated if t in kg_tags]
    eliminated = [t for t in operated if t not in kg_tags]
    rows.append({
        "cluster_id": cluster_id,
        "num_operated_tags": len(operated),
        "num_kept_tags": len(kept),
        "num_eliminated_tags": len(eliminated),
        "operated_tags": ", ".join(operated),
        "kept_tags_after_kg_filter": ", ".join(kept),
        "eliminated_tags": ", ".join(eliminated),
    })

episode_tag_filter = pd.DataFrame(rows).sort_values("cluster_id").reset_index(drop=True)

# ── Global kept / eliminated tags across all episodes ──────────────────────────
kept_global = sorted(t for t in all_operated_tags if t in kg_tags)
eliminated_global = sorted(t for t in all_operated_tags if t not in kg_tags)

elim_ca = ca[ca["Source"].isin(eliminated_global)]
eliminated_summary = (
    elim_ca.groupby("Source")
    .agg(num_actions=("Source", "size"), num_episodes=("cluster_id", "nunique"))
    .reset_index()
    .rename(columns={"Source": "eliminated_tag"})
    .sort_values(["num_episodes", "num_actions"], ascending=False)
    .reset_index(drop=True)
)

# ── Save Excel (per-episode filtering + global eliminated summary) ─────────────
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    episode_tag_filter.to_excel(writer, sheet_name="episode_tag_filtering", index=False)
    eliminated_summary.to_excel(writer, sheet_name="eliminated_tags_summary", index=False)

print(f"Saved: {OUTPUT_XLSX}")
print(f"  Sheet 'episode_tag_filtering'   : {len(episode_tag_filter):,} episodes")
print(f"  Sheet 'eliminated_tags_summary' : {len(eliminated_summary):,} eliminated tags\n")
print(f"Unique operated tags (all episodes): {len(all_operated_tags):,}")
print(f"  Kept  (KG-relevant)              : {len(kept_global):,}")
print(f"  Eliminated (not KG-relevant)     : {len(eliminated_global):,}\n")
print(f"Avg operated tags / episode  : {episode_tag_filter['num_operated_tags'].mean():.1f}")
print(f"Avg kept tags / episode      : {episode_tag_filter['num_kept_tags'].mean():.1f}")
print(f"Avg eliminated tags / episode: {episode_tag_filter['num_eliminated_tags'].mean():.1f}")
display(episode_tag_filter.head(10))


Saved: /home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_kg_filtered_control_action_tags_per_episode.xlsx
  Sheet 'episode_tag_filtering'   : 525 episodes
  Sheet 'eliminated_tags_summary' : 137 eliminated tags

Unique operated tags (all episodes): 195
  Kept  (KG-relevant)              : 58
  Eliminated (not KG-relevant)     : 137

Avg operated tags / episode  : 8.0
Avg kept tags / episode      : 4.9
Avg eliminated tags / episode: 3.2


,cluster_id,num_operated_tags,num_kept_tags,num_eliminated_tags,operated_tags,kept_tags_after_kg_filter,eliminated_tags
0,1,16,13,3,"03FIC_1085, 03FIC_3227, 03FIC_3435, 03HIC_1141...","03FIC_1085, 03FIC_3227, 03FIC_3435, 03HIC_1141...","03HIC_3100, 03XAX_1001, 03XAX_1002"
1,2,9,2,7,"03FIC_3227, 03FIC_3435, 03GHS_0121A, 03GHS_012...","03FIC_3227, 03FIC_3435","03GHS_0121A, 03GHS_0121AA, 03GHS_0121B, 03GM_0..."
2,3,7,3,4,"03FIC_3227, 03FIC_3435, 03HIC_3100, 03LIC_1034...","03FIC_3227, 03FIC_3435, 03LIC_1034","03HIC_3100, 03XAX_1001, 03XAX_1002, 03XAX_1428OR"
3,4,8,4,4,"03FIC_3227, 03FIC_3435, 03HIC_3100, 03LIC_1016...","03FIC_3227, 03FIC_3435, 03LIC_1016, 03LIC_1034","03HIC_3100, 03XAX_1001, 03XAX_1002, 03XAX_1428OR"
4,5,10,4,6,"03C153_PM, 03FIC_3227, 03FIC_3435, 03HIC_3100,...","03FIC_3227, 03FIC_3435, 03LIC_1016, 03LIC_1034","03C153_PM, 03HIC_3100, 03KV_3213, 03XAX_1001, ..."
5,6,2,2,0,"03FIC_3435, 03HIC_3252B","03FIC_3435, 03HIC_3252B",
6,7,1,1,0,03FIC_3435,03FIC_3435,
7,8,2,2,0,"03FIC_3435, 03HIC_1141","03FIC_3435, 03HIC_1141",
8,9,2,2,0,"03FIC_3435, 03LIC_1031","03FIC_3435, 03LIC_1031",
9,10,2,1,1,"03FIC_3435, 03HIC_3100",03FIC_3435,03HIC_3100


---
## All Tags Eliminated by KG Filtering

Every `Source` tag that was operated in at least one episode but is **not** present in the
`03LIC_1071` KG tag list — i.e. the tags removed by the KG-path filter — with the number of
episodes and total actions in which each appeared.


In [4]:
# ── All tags eliminated by the KG filter (across all episodes) ─────────────────
print(f"Total tags eliminated across all episodes: {len(eliminated_global)}\n")
print(eliminated_global)

display(Markdown(f"### {len(eliminated_summary)} Eliminated Tags (ranked by episodes & actions)"))
display(
    eliminated_summary.style.set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]},
    ]).background_gradient(subset=["num_episodes", "num_actions"], cmap="Reds")
)


Total tags eliminated across all episodes: 137

['02EDPV_1052', '02EDPV_1094', '02FAX_1247', '02FIC_1247', '02HIC_1050', '02HIC_1087', '02KM_1139_ST', '02LI_1041_MOS', '02LI_1085_MOS', '02PDC_1220_MAN', '02SDV_1050A', '02SDV_1183', '02SDV_1184', '02SDV_1239', '02SDV_1250', '02TIC_1282', '02XAX_1003', '02XAX_1004', '02XAX_1005', '02XAX_1007', '02XAX_1011', '02XAX_1012', '02XAX_1029_SEL', '02XAX_1090_SEL', '02XAX_1092_SEL', '02XAX_1093_SEL', '02XAX_1095_SEL', '02XAX_1143_RST', '02XAX_1220_AUT', '02XAX_1220_MAN', '03C153_PM', '03EHS_1511BA', '03EHS_1511D', '03EM_1511A', '03EM_1511B', '03ESDV_3246', '03FIC_3435A', '03GHS_0112A', '03GHS_0112AA', '03GHS_0121A', '03GHS_0121AA', '03GHS_0121B', '03GHS_0151AA', '03GHS_0151BA', '03GHS_0152AA', '03GHS_0152BA', '03GHS_0153BA', '03GHS_0153CA', '03GHS_1511AA', '03GHS_1511AB', '03GHS_1511BA', '03GM_0112', '03GM_0112A', '03GM_0114', '03GM_0114A', '03GM_0121', '03GM_0121A', '03GM_0151A', '03GM_0151B', '03GM_0152A', '03GM_0152B', '03GM_0153A', '03GM_0153

### 137 Eliminated Tags (ranked by episodes & actions)

,eliminated_tag,num_actions,num_episodes
0,03C153_PM,441,185
1,03HIC_3100,3518,180
2,02HIC_1087,1925,84
3,02HIC_1050,1535,81
4,02FIC_1247,649,68
5,03TIC_0153,64,60
6,03ESDV_3246,51,40
7,03KV_3213,52,32
8,03KV_3210,31,31
9,03KV_3212,39,30


---
# Part 2 — KG-Filtered Operated Tags + Action Details per SME Episode (SIT-Validation Workbook)

Populate `DATA/ADNOCGAS_Plant3Train1_RCA -After SIT Validation.xlsx` with, for **every SME-reviewed
episode row**, four columns:
1. **KG-filtered operated tags** — control-action tags in the ±30 min window that are **knowledge-graph
   relevant** to that alarm tag.
2. **Eliminated tags** — operated tags removed by the KG filter (not on the alarm tag's paths).
3. **Action details** — for the **KG-kept tags only**, each tag's type / direction / step magnitude, in chronological order.
4. **Mapping comments** — flags any date/Episode-No inconsistency.

**KG filtering** (new): from the first sheet of
`DATA/Full_DB_Merged_Tag_Instrument_Sequences_V17Input 1.xlsx`, column `DCS_Path` (comma-separated full
tag names). For each alarm tag we keep only the paths that **contain that tag**, take the **union of all
tags** on those paths → the alarm tag's KG-relevant set, then split each episode's operated tags into
**kept** (in the set) vs **eliminated** (not in the set).

**Window** = *alarm start − 30 min* → *alarm end + 30 min* (filtered from the −4 h/+1 h `control_actions`
sheet by `VT_Start`). **Action details** = one line per **KG-kept** tag (ordered by first-action time,
numbered + timestamped); within a tag the moves are in time order (MODE/valve state changes shown where
they occur), consecutive same type+direction numeric moves collapse to `start→end (net, #moves)`; OP in
**%**, SP in engineering units; `OP`/`SP`/`MODE` only.

**Row → episode mapping** (layered, handles the messy workbook): `EPI_DATE` (Episode No = cluster_id, date
within 2 days) → `EPI_TOD` (Episode No = cluster_id, time-of-day matches → recovers date typos) → `TIME`
(Date+time matches a cluster start) → `EPI_ONLY` → `NO_MATCH`.

Four columns are appended per tag sheet; the file is rebuilt from a **pristine backup** each run so no
other cell/sheet is affected. Tag sheets: `1071` (×2), `PIC_1104` (×2), `TIC_1009`, `LIC_1016`,
`TIC_1023`, `LIC_1619`. Skipped: `Read me`, `unified_validation_results-Fals`.


In [16]:
import datetime as _dt
import pandas as pd

# ── Inputs ─────────────────────────────────────────────────────────────────────
# Read the SME rows from the pristine original workbook; write the KG-filtered deliverable to a new file.
# (The user renamed the earlier non-KG output to "..._without_kg.xlsx"; this produces the "..._with_kg.xlsx".)
SIT_SOURCE = "/home/h604827/ControlActions/DATA/ADNOCGAS_Plant3Train1_RCA -After SIT Validation_ORIGINAL_backup.xlsx"
SIT_OUTPUT = "/home/h604827/ControlActions/DATA/ADNOCGAS_Plant3Train1_RCA -After SIT Validation_with_control_actions_with_kg.xlsx"
KG_XLSX  = "/home/h604827/ControlActions/DATA/Full_DB_Merged_Tag_Instrument_Sequences_V17Input 1.xlsx"

# tag -> clustered control-actions workbook (sheets 'alarm_clusters' + 'control_actions')
RES = {
    '1071': "/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx",
    '1104': "/home/h604827/ControlActions/RESULTS/03PIC_1104_PVHI_episodes_12JUN2026_1451/03PIC_1104_pvhi_alarms_clustered_with_control_actions.xlsx",
    '1009': "/home/h604827/ControlActions/RESULTS/03TIC_1009_PVLO_episodes_14JUN2026_1140/03TIC_1009_pvlo_alarms_clustered_with_control_actions.xlsx",
    '1016': "/home/h604827/ControlActions/RESULTS/03LIC_1016_episodes_02JUN2026_1607/03LIC_1016_pvlo_alarms_clustered_with_control_actions.xlsx",
    '1023': "/home/h604827/ControlActions/RESULTS/03TIC_1023_episodes_03JUN2026_0915/03TIC_1023_pvlo_alarms_clustered_with_control_actions.xlsx",
    '1619': "/home/h604827/ControlActions/RESULTS/03LIC_1619_PVLO_episodes_26JUN2026_1130/03LIC_1619_pvlo_alarms_clustered_with_control_actions.xlsx",
}

# sheet name -> (tag, date_col_idx, time_col_idx, episode_col_idx or None)   [header = row 0]
SHEET_CFG = {
    '1071 -Updated 2':            ('1071', 2, 3, 1),
    'Updated LIC_1071_forSITVal': ('1071', 2, 3, 1),
    ' PIC_1104':                  ('1104', 1, 2, None),
    'PIC1104-Upadated2':          ('1104', 1, 2, None),
    'TIC_1009-Updated ':          ('1009', 1, 2, 0),
    'Updated LIC_1016':           ('1016', 2, 3, 1),
    'Updated TIC_1023':           ('1023', 2, 3, 1),
    'Updated LIC_1619':           ('1619', 2, 3, 1),
}

# alarm tag -> full DCS tag name used to select the KG paths that contain it
TAG_FULLNAME = {
    '1071': '03LIC_1071', '1104': '03PIC_1104', '1009': '03TIC_1009',
    '1016': '03LIC_1016', '1023': '03TIC_1023', '1619': '03LIC_1619',
}

KEPT_COL_HEADER    = "KG-filtered operated tags (-30min to +30min of alarm)"
ELIM_COL_HEADER    = "Eliminated tags (not in KG paths of the alarm tag)"
DETAIL_COL_HEADER  = "Action details (per tag, in order)"
COMMENT_COL_HEADER = "Mapping comments"
WINDOW       = pd.Timedelta(minutes=30)
TIME_TOL     = pd.Timedelta(minutes=30)
EPI_DATE_TOL = pd.Timedelta(days=2)
NO_MATCH       = "NO_MATCH"
NO_ACTIONS_MSG = "No control actions available within +/-30 mins of alarm"

# ── KG relevance sets ───────────────────────────────────────────────────────────
# Use ONLY the first sheet's DCS_Path column (comma-separated full tag names). For each
# alarm tag, keep the paths that CONTAIN that tag, then take the union of all tags in those
# paths -> the alarm tag's KG-relevant tag set.
_kg_paths = pd.read_excel(KG_XLSX, sheet_name=0, usecols=['DCS_Path'])['DCS_Path'].dropna().astype(str)
_kg_token_sets = _kg_paths.apply(lambda p: {t.strip() for t in p.split(',') if t.strip()})
_kg_cache = {}
def kg_relevant_tags(tag):
    full = TAG_FULLNAME[tag]
    if full not in _kg_cache:
        s = set()
        for ts in _kg_token_sets[_kg_token_sets.apply(lambda x: full in x)]:
            s |= ts
        _kg_cache[full] = s
    return _kg_cache[full]

print("KG-relevant tag-set size per alarm tag (paths containing that tag):")
for _t, _f in TAG_FULLNAME.items():
    print(f"  {_f}: {len(kg_relevant_tags(_t))} tags")

def _load_tag(tag):
    p = RES[tag]
    ac = pd.read_excel(p, sheet_name='alarm_clusters')
    ac['cluster_start_time'] = pd.to_datetime(ac['cluster_start_time'])
    ac['cluster_end_time']   = pd.to_datetime(ac['cluster_end_time'])
    cb = ac.groupby('cluster_id').agg(cstart=('cluster_start_time', 'min'),
                                      cend=('cluster_end_time', 'max'))
    ca = pd.read_excel(p, sheet_name='control_actions',
                       usecols=['cluster_id', 'Source', 'Description', 'action_direction',
                                'PrevValue', 'Value', 'VT_Start'])
    ca['VT_Start'] = pd.to_datetime(ca['VT_Start'])
    ca = ca[ca['Source'].notna()].copy()
    return cb, ca

def _parse_time(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return None
    if isinstance(t, _dt.time):
        return t
    if isinstance(t, (pd.Timestamp, _dt.datetime)):
        return t.time()
    ts = pd.to_datetime(str(t), errors='coerce')
    return None if pd.isna(ts) else ts.time()

def _as_int(x):
    try:
        return int(float(x))
    except (ValueError, TypeError):
        return None

def _map_row(rdt, epi, has_time, cb):
    e = _as_int(epi)
    if e is not None and e not in cb.index:
        e = None
    if e is not None and abs(cb.at[e, 'cstart'] - rdt) <= EPI_DATE_TOL:
        return e, 'EPI_DATE'
    if e is not None and has_time and cb.at[e, 'cstart'].strftime('%H:%M') == rdt.strftime('%H:%M'):
        return e, 'EPI_TOD'
    j = (cb['cstart'] - rdt).abs().idxmin()
    if abs(cb.at[j, 'cstart'] - rdt) <= TIME_TOL:
        return j, 'TIME'
    if e is not None:
        return e, 'EPI_ONLY'
    return None, NO_MATCH

def _comment(method, rdt, has_time, epi, cid, cb):
    cs = cb.at[cid, 'cstart']; e = _as_int(epi)
    if method == 'EPI_DATE':
        if has_time and cs.strftime('%H:%M') != rdt.strftime('%H:%M') and abs(cs - rdt) > pd.Timedelta(minutes=30):
            return (f"Time typo: sheet time {rdt.strftime('%H:%M')} vs actual alarm "
                    f"{cs.strftime('%Y-%m-%d %H:%M')} (same date); mapped via Episode No {e}=cluster {cid}.")
        return ""
    if method == 'EPI_TOD':
        return (f"Date typo: sheet date {rdt.strftime('%Y-%m-%d')} has no alarm; actual alarm "
                f"{cs.strftime('%Y-%m-%d %H:%M')} (time-of-day matches); mapped via Episode No {e}=cluster {cid}.")
    if method == 'TIME':
        if e is not None and e != cid:
            return (f"Episode-No typo: sheet says {e}, but Date+time matches cluster {cid} "
                    f"({cs.strftime('%Y-%m-%d %H:%M')}); mapped via Date+time.")
        return ""
    if method == 'EPI_ONLY':
        return (f"Weak match: mapped via Episode No {e}=cluster {cid} only "
                f"(cluster start {cs.strftime('%Y-%m-%d %H:%M')}; sheet date & time both differ) - verify.")
    return ""

def _nomatch_comment(rdt, has_time, cb):
    msg = "No matching alarm cluster (likely date-entry error in sheet)."
    if has_time:
        same = cb[cb['cstart'].dt.strftime('%H:%M') == rdt.strftime('%H:%M')]
        if len(same):
            j = (same['cstart'] - rdt).abs().idxmin()
            msg += f" Nearest cluster with same time-of-day: {same.at[j, 'cstart'].strftime('%Y-%m-%d %H:%M')} (cluster {j})."
    return msg

# ── Action-detail rendering: type + direction + step magnitude, in chronological order ──
_ARROW = {"increase": "↑", "decrease": "↓"}
_UNIT  = {"OP": "%", "SP": ""}

def _is_num(x):
    try:
        float(x); return True
    except (ValueError, TypeError):
        return False

def _fmt_val(x):
    return f"{round(float(x), 2):g}"

def _action_detail(win):
    df = win[win['Description'].isin(['OP', 'SP', 'MODE'])].sort_values('VT_Start')
    if df.empty:
        return ""
    first_time = df.groupby('Source')['VT_Start'].min().sort_values()
    lines = []
    for seq, (tag, t0) in enumerate(first_time.items(), 1):
        rows = df[df['Source'] == tag].sort_values('VT_Start').to_dict('records')
        segs = []; i = 0
        while i < len(rows):
            r = rows[i]; desc = r['Description']; d = r['action_direction']
            if desc == 'MODE':
                segs.append(f"MODE {r['PrevValue']}→{r['Value']}"); i += 1
            elif _is_num(r['PrevValue']) and _is_num(r['Value']):
                j = i; last = r
                while (j + 1 < len(rows) and rows[j+1]['Description'] == desc
                       and rows[j+1]['action_direction'] == d
                       and _is_num(rows[j+1]['PrevValue']) and _is_num(rows[j+1]['Value'])):
                    j += 1; last = rows[j]
                start = float(r['PrevValue']); end = float(last['Value']); n = j - i + 1
                net = round(end - start, 2); unit = _UNIT.get(desc, ""); ar = _ARROW.get(d, "")
                mv = f"{n} move" + ("s" if n > 1 else "")
                segs.append(f"{desc}{(' ' + ar) if ar else ''} {_fmt_val(start)}→{_fmt_val(end)} ({net:+g}{unit}, {mv})")
                i = j + 1
            else:
                j = i; last = r
                while (j + 1 < len(rows) and rows[j+1]['Description'] == desc
                       and not (_is_num(rows[j+1]['PrevValue']) and _is_num(rows[j+1]['Value']))):
                    j += 1; last = rows[j]
                n = j - i + 1
                segs.append(f"{desc} {r['PrevValue']}→{last['Value']}" + (f" ({n}x)" if n > 1 else ""))
                i = j + 1
        lines.append(f"{seq}. [{t0.strftime('%H:%M')}] {tag} — " + ", then ".join(segs))
    return "\n".join(lines)

# ── Compute KG-filtered kept/eliminated tags + action details + comment per row ─
_tag_cache = {}
sheet_kept, sheet_eliminated, sheet_details, sheet_comments = {}, {}, {}, {}
report_rows = []

for sheet, (tag, dcol, tcol, ecol) in SHEET_CFG.items():
    if tag not in _tag_cache:
        _tag_cache[tag] = _load_tag(tag)
    cb, catbl = _tag_cache[tag]
    kg_set = kg_relevant_tags(tag)
    raw = pd.read_excel(SIT_SOURCE, sheet_name=sheet, header=None)
    kept_map, elim_map, det_map, com_map = {}, {}, {}, {}
    for idx in range(1, len(raw)):
        dval = pd.to_datetime(raw.iat[idx, dcol], errors='coerce')
        if pd.isna(dval):
            continue
        tpart = _parse_time(raw.iat[idx, tcol]); has_time = tpart is not None
        rdt = pd.Timestamp.combine(dval.date(), tpart) if has_time else dval
        epi = raw.iat[idx, ecol] if ecol is not None else None
        cid, method = _map_row(rdt, epi, has_time, cb)
        if cid is None:
            kept_map[idx] = NO_MATCH; elim_map[idx] = ""; det_map[idx] = ""
            com_map[idx] = _nomatch_comment(rdt, has_time, cb)
            report_rows.append((sheet, tag, idx, rdt, None, method, 0, 0, 0))
            continue
        cs, ce = cb.at[cid, 'cstart'], cb.at[cid, 'cend']
        win = catbl[(catbl['cluster_id'] == cid) &
                    (catbl['VT_Start'] >= cs - WINDOW) &
                    (catbl['VT_Start'] <= ce + WINDOW)]
        operated = sorted(win['Source'].unique())
        if not operated:
            kept_map[idx] = NO_ACTIONS_MSG; elim_map[idx] = ""; n_kept = n_elim = 0
        else:
            kept = [t for t in operated if t in kg_set]
            elim = [t for t in operated if t not in kg_set]
            kept_map[idx] = ", ".join(kept)     # blank if none KG-relevant
            elim_map[idx] = ", ".join(elim)     # blank if none eliminated
            n_kept, n_elim = len(kept), len(elim)
        det_map[idx] = _action_detail(win[win['Source'].isin(kg_set)])   # KG-kept tags only
        com_map[idx] = _comment(method, rdt, has_time, epi, cid, cb)
        report_rows.append((sheet, tag, idx, rdt, cid, method, len(operated), n_kept, n_elim))
    sheet_kept[sheet] = kept_map
    sheet_eliminated[sheet] = elim_map
    sheet_details[sheet] = det_map
    sheet_comments[sheet] = com_map

report = pd.DataFrame(report_rows,
                      columns=['sheet', 'tag', 'excel_row', 'row_datetime', 'cluster_id',
                               'method', 'n_operated', 'n_kept', 'n_eliminated'])
print("\nPer-sheet summary:")
print(report.groupby('sheet').agg(rows=('excel_row', 'size'),
                                  mapped=('cluster_id', lambda s: s.notna().sum()),
                                  no_match=('method', lambda s: (s == NO_MATCH).sum()),
                                  operated=('n_operated', 'sum'),
                                  kept=('n_kept', 'sum'),
                                  eliminated=('n_eliminated', 'sum')).to_string())
print(f"\nAcross all rows: operated tag-instances={report['n_operated'].sum()}, "
      f"kept={report['n_kept'].sum()}, eliminated={report['n_eliminated'].sum()}")
print(f"mapped={report['cluster_id'].notna().sum()}  NO_MATCH={(report['method'] == NO_MATCH).sum()}")

_ex = report[(report['method'] != NO_MATCH) & (report['n_operated'] > 0)]
if len(_ex):
    s = _ex.iloc[0]['sheet']; ix = int(_ex.iloc[0]['excel_row'])
    print(f"\nSample — sheet {s!r}, row {ix}:")
    print("  KEPT      :", sheet_kept[s][ix])
    print("  ELIMINATED:", sheet_eliminated[s][ix])
    print("  DETAILS   :")
    for ln in sheet_details[s][ix].split("\n"):
        print("     ", ln)


KG-relevant tag-set size per alarm tag (paths containing that tag):
  03LIC_1071: 908 tags
  03PIC_1104: 840 tags
  03TIC_1009: 698 tags
  03LIC_1016: 851 tags
  03TIC_1023: 666 tags
  03LIC_1619: 627 tags

Per-sheet summary:
                            rows  mapped  no_match  operated  kept  eliminated
sheet                                                                         
 PIC_1104                     10       7         3        93    52          41
1071 -Updated 2               31      30         1        74    54          20
PIC1104-Upadated2             10       7         3        93    52          41
TIC_1009-Updated               7       7         0        51    22          29
Updated LIC_1016              30      30         0        35    34           1
Updated LIC_1071_forSITVal    31      30         1        74    54          20
Updated LIC_1619              21      21         0        37    32           5
Updated TIC_1023              30      30         0        20   

In [27]:
import shutil
import openpyxl
from openpyxl.styles import Alignment

# ── Build the KG-filtered deliverable from the pristine original SME workbook ──────
# Copying the untouched original each run guarantees exactly our 4 columns and leaves every
# other cell/sheet exactly as in the original (no orphaned columns from earlier schemas).
shutil.copy2(SIT_SOURCE, SIT_OUTPUT)
print(f"Created KG-filtered workbook from pristine source:\n  source: {SIT_SOURCE}\n  output: {SIT_OUTPUT}\n")

# ── Append: KG-filtered tags | Eliminated tags | Action details | Mapping comments ─
wb = openpyxl.load_workbook(SIT_OUTPUT)
for sheet in SHEET_CFG:
    ws = wb[sheet]
    c = ws.max_column + 1                       # 1) KG-filtered (kept) operated tags
    ws.cell(row=1, column=c, value=KEPT_COL_HEADER)
    for idx, val in sheet_kept[sheet].items():
        ws.cell(row=idx + 1, column=c, value=val)
    c = ws.max_column + 1                       # 2) eliminated tags
    ws.cell(row=1, column=c, value=ELIM_COL_HEADER)
    for idx, val in sheet_eliminated[sheet].items():
        ws.cell(row=idx + 1, column=c, value=val)
    c = ws.max_column + 1                       # 3) action details (multi-line, wrapped)
    ws.cell(row=1, column=c, value=DETAIL_COL_HEADER)
    for idx, val in sheet_details[sheet].items():
        cell = ws.cell(row=idx + 1, column=c, value=val)
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    ws.column_dimensions[ws.cell(row=1, column=c).column_letter].width = 70
    c = ws.max_column + 1                       # 4) mapping comments
    ws.cell(row=1, column=c, value=COMMENT_COL_HEADER)
    for idx, val in sheet_comments[sheet].items():
        ws.cell(row=idx + 1, column=c, value=val)
    print(f"  {sheet!r}: wrote KG-filtered | Eliminated | Action details | Comments")

wb.save(SIT_OUTPUT)
print(f"\nSaved: {SIT_OUTPUT}")


Created KG-filtered workbook from pristine source:
  source: /home/h604827/ControlActions/DATA/ADNOCGAS_Plant3Train1_RCA -After SIT Validation_ORIGINAL_backup.xlsx
  output: /home/h604827/ControlActions/DATA/ADNOCGAS_Plant3Train1_RCA -After SIT Validation_with_control_actions_with_kg.xlsx

  '1071 -Updated 2': wrote KG-filtered | Eliminated | Action details | Comments
  'Updated LIC_1071_forSITVal': wrote KG-filtered | Eliminated | Action details | Comments
  ' PIC_1104': wrote KG-filtered | Eliminated | Action details | Comments
  'PIC1104-Upadated2': wrote KG-filtered | Eliminated | Action details | Comments
  'TIC_1009-Updated ': wrote KG-filtered | Eliminated | Action details | Comments
  'Updated LIC_1016': wrote KG-filtered | Eliminated | Action details | Comments
  'Updated TIC_1023': wrote KG-filtered | Eliminated | Action details | Comments
  'Updated LIC_1619': wrote KG-filtered | Eliminated | Action details | Comments

Saved: /home/h604827/ControlActions/DATA/ADNOCGAS_Plant3

---
# Part 3 — Ground-Truth Cause-Tag Pre-Alarm Trend (90 min) — all SME-reviewed loops

For every SME-reviewed episode across **all six alarm loops** (`1071`, `1104`, `1009`, `1016`,
`1023`, `1619`), characterise how each **ground-truth cause tag** (`Cause 1/2/3`) was behaving in
the **90 minutes before the alarm** — *increasing*, *decreasing*, or *stagnant* — robustly, so
short-window noise doesn't fool us and a late-developing move over the long window isn't washed out.

**Anchor** = cluster start (alarm instant). **Window** = `[alarm − 90 min, alarm]`, no look-ahead.
**Signal** = each cause tag's `.PV`, read **loop-export-first** (complete) with the all-loops merged
historian as gap-fill (the merged export has scattered per-tag gaps).

### Method (multi-scale + robust)
1. **Denoise** — 5-min rolling *median* (kills minute spikes without erasing drift); estimate the
   tag's in-window **noise** from the residual (robust MAD).
2. **Trend at several scales** — robust **Theil–Sen** slope over the last **15 / 30 / 60 / 90 min**
   (looking at all scales *is* the fix for "short misses noise, long misses detail"), plus a
   **Mann–Kendall** monotonic-trend test over the full window.
3. **Dead-band classification** — a move counts as real only if `|net change| > max(5%·operating-band,
   2·noise)`; otherwise **stagnant**. The band is *per-tag*, from `operating_limits.csv` + the tag's noise.
4. **Onset + character** — locate when the final move began and label the shape: `strong_ramp`,
   `late_move`, `reversing`, `oscillating`, `weak_drift`, or `flat`.

### Outputs
- **In the `_kg` workbook** (every tag sheet): the behaviour is written **into each cause cell** next to
  the tag name (e.g. `FI_1000 [decreasing]`). _(The separate multi-line detail column was removed on
  request — the full per-tag metrics remain in the CSV below.)_
- **Tidy table** `gt_trend_long` (+ CSV) — one row per *(loop × episode × cause tag)* with every metric.
- **`plot_gt_trend(i)`** — visual validation of any row.

_Cause columns are **auto-detected per sheet** (only the `Cause 1/2/3` block). Requires the Part 2
cells above (reuses `_map_row`, `_parse_time`, `_load_tag`, `SHEET_CFG`, `SIT_SOURCE`, `SIT_OUTPUT`)._


In [13]:
# ═══ Part 3.1 — config, cause-tag resolver, load pre-alarm PV context ═══════════
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as _pq
from scipy.stats import theilslopes, norm
from IPython.display import display

# PV sources: PREFER each loop's complete PV-OP export; fall back to the all-loops merged
# export (which has scattered per-tag gaps, e.g. 02FI_1000.PV / 03FIC_3435.PV NaN on 2025-05-25).
_PVOP = "/home/h604827/ControlActions/DATA/PV-OP_data"
LOOP_PARQUETS = {
    '1071': f"{_PVOP}/03LIC_1071_JAN_2026.parquet",
    '1104': f"{_PVOP}/03PIC_1104_JAN_2026.parquet",
    '1009': f"{_PVOP}/03TIC_1009_JUNE_2026.parquet",
    '1016': f"{_PVOP}/03LIC_1016_JAN_2026.parquet",
    '1023': f"{_PVOP}/03TIC_1023_JAN_2026.parquet",
    '1619': f"{_PVOP}/03LIC_1619_MAY_2026.parquet",
}
MERGED_PARQUET = "/home/h604827/ControlActions/DATA/new_rca_pv_op_data/merged_all_historian_tags.parquet"
TAGLIST_CSV    = "/home/h604827/ControlActions/DATA/new_rca_pv_op_data/merged_all_historian_tags_taglist.csv"
OPLIM_CSV      = "/home/h604827/ControlActions/DATA/operating_limits.csv"

PRE_MIN       = 90                  # minutes of pre-alarm context
WINDOWS       = [15, 30, 60, 90]    # multi-scale trend windows (min, ending at alarm)
SMOOTH_MIN    = 5                   # rolling-median smoothing window (min)
DEADBAND_FRAC = 0.05               # dead-band = 5% of the tag's operating-limit span ...
NOISE_K       = 2.0                # ... or k x in-window noise, whichever is larger
STRONG_MULT   = 2.0               # |net90| >= STRONG_MULT x dead-band => strong ramp

# ── Resolver: loose GT token ("PIC_1013", "FI_1000", "03HIC_1009A") -> real ".PV" column ──
_tl = pd.read_csv(TAGLIST_CSV)
_pv_tags = _tl[_tl['signal_type'] == 'PV'].copy()
_ol = pd.read_csv(OPLIM_CSV).set_index('TAG_NAME')
_TOK = re.compile(r'([A-Z]{2,3})[ _]*(\d{3,4}[A-Z]?)')

def resolve_pv_tags(cell):
    """Return de-duplicated [(token, pv_col)] for each tag-like token in a cause cell."""
    if not isinstance(cell, str):
        return []
    seen, out = set(), []
    for instr, num in _TOK.findall(cell.upper()):
        cand = _pv_tags[(_pv_tags['instrument_type'] == instr) &
                        (_pv_tags['base_tag'].str.endswith('_' + num))]
        tags = sorted(cand['tag'].tolist())
        pref = [t for t in tags if t.startswith('03')] or tags   # prefer plant-03
        if pref and pref[0] not in seen:
            seen.add(pref[0])
            out.append((f"{instr}_{num}", pref[0]))
    return out

# ── Cause columns per sheet: only the "Cause 1/2/3" block (row 0 or the sub-header row 1) ──
_CAUSE_HDR = re.compile(r'cause\s*-?\s*[123]\b', re.I)   # matches "Cause 1", "Cause -2", "Cause-3"
def _detect_cause_cols(raw):
    cols = []
    for c in range(raw.shape[1]):
        for r in (0, 1):
            v = raw.iat[r, c] if r < len(raw) else None
            if isinstance(v, str) and _CAUSE_HDR.search(v):
                cols.append(c); break
    return cols

# ── Parse ALL SME sheets: map each episode row -> cluster, collect cause tags ──
_cb_cache = {}
def _cb(tag):
    if tag not in _cb_cache:
        _cb_cache[tag] = _load_tag(tag)[0]     # cluster bounds (cstart/cend) per cluster_id
    return _cb_cache[tag]

gt_records, sheet_cause_cols, _needed_pv = [], {}, set()
for sheet, (tag, dcol, tcol, ecol) in SHEET_CFG.items():
    raw = pd.read_excel(SIT_SOURCE, sheet_name=sheet, header=None)
    sheet_cause_cols[sheet] = _detect_cause_cols(raw)
    cb = _cb(tag)
    for idx in range(1, len(raw)):             # skip main header; date-parse skips sub-headers
        dval = pd.to_datetime(raw.iat[idx, dcol], errors='coerce')
        if pd.isna(dval):
            continue
        tpart = _parse_time(raw.iat[idx, tcol]); has_time = tpart is not None
        rdt = pd.Timestamp.combine(dval.date(), tpart) if has_time else dval
        epi = raw.iat[idx, ecol] if ecol is not None else None
        cid, method = _map_row(rdt, epi, has_time, cb)
        if cid is None:
            continue
        cstart = cb.at[cid, 'cstart']
        for slot, col in enumerate(sheet_cause_cols[sheet], start=1):
            cell = raw.iat[idx, col]
            for token, pv_col in resolve_pv_tags(cell):
                _needed_pv.add(pv_col)
                gt_records.append(dict(
                    sheet=sheet, alarm_tag=tag, excel_row=idx, cause_col=col, cause_slot=slot,
                    cluster_id=cid, cstart=cstart, gt_token=token, pv_col=pv_col,
                    original_text=str(cell).strip()))

print("Cause columns auto-detected per sheet:")
for _sh, _cc in sheet_cause_cols.items():
    print(f"  {_sh!r}: {_cc}")

# ── Load needed PV columns: merged base, each loop export overwrites (loop wins over gaps) ──
def _read_avail(path, cols):
    have = [c for c in cols if c in set(_pq.read_schema(path).names)]
    if not have:
        return None
    df = pd.read_parquet(path, columns=['TimeStamp', *have])
    df = df.dropna(subset=['TimeStamp']).set_index('TimeStamp').sort_index()
    return df[~df.index.duplicated(keep='last')]

_need = sorted(_needed_pv)
pv_df = _read_avail(MERGED_PARQUET, _need)          # base: all needed cols, but gappy
if pv_df is None:
    pv_df = pd.DataFrame(index=pd.DatetimeIndex([], name='TimeStamp'))
for c in _need:                                     # ensure every needed col exists
    if c not in pv_df.columns:
        pv_df[c] = np.nan
pv_df = pv_df.astype('float64')                     # unify dtype (merged float32 vs loop float64)
for _t, _p in LOOP_PARQUETS.items():                # loop exports overwrite with their (non-NaN) data
    _ldf = _read_avail(_p, _need)
    if _ldf is not None:
        pv_df.update(_ldf.astype('float64'))

_allnan = [c for c in _need if pv_df[c].notna().sum() == 0]
print(f"\nGT cause-tag records (loop x episode x cause tag): {len(gt_records)}")
print(f"Distinct PV columns needed: {len(_need)}")
print(f"PV frame: {pv_df.shape[0]:,} rows x {pv_df.shape[1]} cols "
      f"({pv_df.index.min()} -> {pv_df.index.max()})")
print(f"PV cols with NO data at all (-> no_data): {_allnan}")
print(f"PV cols WITHOUT operating limits (noise-only dead-band): "
      f"{sorted(t for t in _need if t not in _ol.index)}")


Cause columns auto-detected per sheet:
  '1071 -Updated 2': [4, 5, 6]
  'Updated LIC_1071_forSITVal': [4, 5, 6]
  ' PIC_1104': [3, 4, 5]
  'PIC1104-Upadated2': [3, 4, 5]
  'TIC_1009-Updated ': [3, 4, 5]
  'Updated LIC_1016': [4, 5, 6]
  'Updated TIC_1023': [4, 5, 6]
  'Updated LIC_1619': [4, 5, 6]

GT cause-tag records (loop x episode x cause tag): 266
Distinct PV columns needed: 20
PV frame: 2,102,317 rows x 20 cols (2022-01-03 22:45:00 -> 2026-01-09 23:29:00)
PV cols with NO data at all (-> no_data): []
PV cols WITHOUT operating limits (noise-only dead-band): ['03FIC_3435.PV', '03HIC_1009A.PV', '03HIC_1009B.PV', '03LIC_1619.PV', '03LI_1196.PV', '03PIC_1023.PV', '03TIC_1009.PV', '03TIC_1023.PV', '03TI_1005.PV', '03TI_1405.PV']


In [24]:
# ═══ Part 3.2 — trend engine: denoise -> multi-scale slope -> dead-band ═════════
def _theil_slope(sub):
    """Theil-Sen slope (EU per minute) of a time-indexed Series; NaN if too few pts."""
    sub = sub.dropna()
    if len(sub) < 5:
        return np.nan
    x = (sub.index - sub.index[0]).total_seconds().to_numpy() / 60.0
    return theilslopes(sub.to_numpy(), x)[0]

def _mann_kendall(y):
    """Non-parametric monotonic-trend test. Returns (label, p_value)."""
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    n = len(y)
    if n < 6:
        return ('insufficient', np.nan)
    s_stat = 0.0
    for i in range(n - 1):
        s_stat += np.sign(y[i + 1:] - y[i]).sum()
    var = n * (n - 1) * (2 * n + 5) / 18.0
    z = (s_stat - np.sign(s_stat)) / np.sqrt(var)
    p = 2 * (1 - norm.cdf(abs(z)))
    if p < 0.05 and z > 0:
        return ('increasing', float(p))
    if p < 0.05 and z < 0:
        return ('decreasing', float(p))
    return ('no-trend', float(p))

def _robust_sigma(resid):
    resid = resid[~np.isnan(resid)]
    if len(resid) < 3:
        return np.nan
    return 1.4826 * np.median(np.abs(resid - np.median(resid)))

def characterize_trend(pv_col, cstart):
    """Characterise a cause tag's PV over the 90 min before the alarm (cstart)."""
    anchor = pd.Timestamp(cstart).floor('min')
    grid = pd.date_range(anchor - pd.Timedelta(minutes=PRE_MIN), anchor, freq='1min')
    s = pv_df[pv_col].reindex(grid, method='nearest', tolerance=pd.Timedelta('30s'))
    coverage = float(s.notna().mean())

    out = dict(pv_col=pv_col, coverage=coverage, direction='no_data',
               trend_character='no_data', data_quality='insufficient_data',
               onset_time=pd.NaT)
    if s.notna().sum() < 10:
        return out

    # denoise (robust) + noise level from the residual
    s_filled = s.interpolate('linear', limit=SMOOTH_MIN, limit_direction='both')
    s_s = s_filled.rolling(SMOOTH_MIN, center=True, min_periods=2).median()
    sigma = _robust_sigma((s - s_s).to_numpy())

    # per-tag operating band + dead-band
    if pv_col in _ol.index:
        lo, hi = float(_ol.at[pv_col, 'LOWER_LIMIT']), float(_ol.at[pv_col, 'UPPER_LIMIT'])
        op_band = hi - lo if hi > lo else np.nan
    else:
        lo = hi = op_band = np.nan
    parts = []
    if np.isfinite(op_band): parts.append(DEADBAND_FRAC * op_band)
    if np.isfinite(sigma):   parts.append(NOISE_K * sigma)
    deadband = max(parts) if parts else max(NOISE_K * float(np.nanstd(s.to_numpy())), 1e-9)

    # multi-window robust slopes + net change (slope x window)
    slopes = {w: _theil_slope(s_s.loc[anchor - pd.Timedelta(minutes=w): anchor]) for w in WINDOWS}
    nets = {w: (slopes[w] * w if np.isfinite(slopes[w]) else np.nan) for w in WINDOWS}
    net90 = nets[90]
    passes = {w: (np.isfinite(nets[w]) and abs(nets[w]) > deadband) for w in WINDOWS}
    signs = {w: (np.sign(nets[w]) if np.isfinite(nets[w]) else 0) for w in WINDOWS}

    # six 15-min segment slopes -> count direction flips (oscillation)
    seg_signs = []
    for k in range(6):
        seg = s_s.loc[anchor - pd.Timedelta(minutes=15 * (k + 1)): anchor - pd.Timedelta(minutes=15 * k)]
        ms = _theil_slope(seg)
        if np.isfinite(ms) and abs(ms * 15) > deadband:
            seg_signs.append(np.sign(ms))
    n_sign_changes = sum(1 for a, b in zip(seg_signs, seg_signs[1:]) if a != b)

    mk_label, mk_p = _mann_kendall(s_s.to_numpy())

    # classify character + direction (priority order)
    active = [signs[w] for w in WINDOWS if passes[w]]
    # a *coherent recent move* = the short/medium windows that clear the dead-band all
    # agree in sign while the full 90-min slope stays flat. That is a genuine late move
    # into the alarm and must be labelled BEFORE 'oscillating' — otherwise earlier
    # in-window wobble (which inflates n_sign_changes) masks a clean one-directional run-up.
    recent_pass_signs = {signs[w] for w in (15, 30, 60) if passes[w]}
    coherent_recent = (not passes[90] and any(passes[w] for w in (15, 30))
                       and len(recent_pass_signs) == 1)
    if not any(passes.values()):
        character, direction = 'flat', 'stagnant'
    elif coherent_recent:
        wp = min(w for w in (15, 30, 60) if passes[w])
        character = 'late_move'
        direction = 'increasing' if nets[wp] > 0 else 'decreasing'
    elif n_sign_changes >= 3 and not passes[90]:
        character, direction = 'oscillating', 'stagnant'
    elif passes[90] and len(set(active)) > 1:
        character = 'reversing'
        direction = 'increasing' if net90 > 0 else 'decreasing'
    elif not passes[90] and any(passes[w] for w in (15, 30)):
        wp = min(w for w in WINDOWS if passes[w])
        character = 'late_move'
        direction = 'increasing' if nets[wp] > 0 else 'decreasing'
    elif passes[90] and abs(net90) >= STRONG_MULT * deadband and mk_label in ('increasing', 'decreasing'):
        character = 'strong_ramp'
        direction = 'increasing' if net90 > 0 else 'decreasing'
    else:
        character = 'weak_drift'
        direction = 'increasing' if net90 > 0 else 'decreasing'

    # onset = last reversal before the final move (peak before a fall / trough before a rise)
    onset_time, from_onset_min, from_onset_eu = pd.NaT, np.nan, np.nan
    vser = s_s.dropna()
    v_alarm = float(vser.iloc[-1]) if len(vser) else np.nan
    v_start = float(vser.iloc[0]) if len(vser) else np.nan
    if direction in ('increasing', 'decreasing') and len(vser):
        onset_time = vser.idxmin() if direction == 'increasing' else vser.idxmax()
        from_onset_min = (anchor - onset_time).total_seconds() / 60.0
        from_onset_eu = v_alarm - float(vser.loc[onset_time])

    pos = ('below_low' if (np.isfinite(lo) and v_alarm < lo)
           else 'above_high' if (np.isfinite(hi) and v_alarm > hi)
           else 'within' if np.isfinite(lo) else 'unknown')

    out.update(
        data_quality=('ok' if coverage >= 0.5 else 'sparse'),
        direction=direction, trend_character=character,
        value_at_alarm=v_alarm, value_90min_ago=v_start,
        net90_eu=net90, pct_of_op_band=(100 * net90 / op_band if np.isfinite(op_band) else np.nan),
        slope15=slopes[15], slope30=slopes[30], slope60=slopes[60], slope90=slopes[90],
        net15=nets[15], net30=nets[30], net60=nets[60],
        mk_trend=mk_label, mk_p=mk_p,
        onset_time=onset_time, from_onset_min=from_onset_min, from_onset_eu=from_onset_eu,
        sigma_noise=sigma, op_band=op_band, deadband_eu=deadband,
        at_alarm_vs_limits=pos, n_sign_changes=n_sign_changes,
    )
    return out

print("Trend engine ready: characterize_trend(pv_col, cstart)")


Trend engine ready: characterize_trend(pv_col, cstart)


In [25]:
# ═══ Part 3.3 — characterise every (loop x episode x cause tag) + tidy table ═══
_char_cache = {}
def _char_cached(pv_col, cstart):
    key = (pv_col, pd.Timestamp(cstart).floor('min'))
    if key not in _char_cache:
        _char_cache[key] = characterize_trend(pv_col, cstart)
    return _char_cache[key]

def _behavior_word(r):
    """Short label written next to the tag inside the cause cell."""
    d = r['direction']
    if d == 'no_data':
        return 'no data'
    if d == 'stagnant':
        return 'stagnant (oscillating)' if r['trend_character'] == 'oscillating' else 'stagnant'
    qual = {'late_move': ' (late)', 'reversing': ' (reversing)', 'weak_drift': ' (weak)'}.get(
        r['trend_character'], '')
    return f"{d}{qual}"

def _detail_line(r):
    """Rich one-liner per cause tag for the workbook detail column."""
    if r['direction'] == 'no_data':
        return f"{r['gt_token']} ({r['pv_col']}): no PV data in window (cov {r['coverage']:.0%})"
    onset = '' if pd.isna(r['onset_time']) else (
        f" | onset {pd.Timestamp(r['onset_time']).strftime('%H:%M')} "
        f"(-{r['from_onset_min']:.0f}min, Δ{r['from_onset_eu']:+.2f})")
    pct = f" ({r['pct_of_op_band']:+.0f}% band)" if np.isfinite(r['pct_of_op_band']) else ''
    return (f"{r['gt_token']} ({r['pv_col']}): {_behavior_word(r)} [{r['trend_character']}] "
            f"| Δ90={r['net90_eu']:+.2f}{pct} "
            f"| slope 15/30/60/90={r['slope15']:+.3f}/{r['slope30']:+.3f}/"
            f"{r['slope60']:+.3f}/{r['slope90']:+.3f} EU/min"
            f"{onset} | at-alarm: {r['at_alarm_vs_limits']} | cov {r['coverage']:.0%}")

rows = []
for rec in gt_records:
    res = _char_cached(rec['pv_col'], rec['cstart'])
    rows.append({**rec, **{k: v for k, v in res.items() if k != 'pv_col'}})
gt_trend_long = pd.DataFrame(rows)
gt_trend_long['behavior'] = gt_trend_long.apply(_behavior_word, axis=1)
gt_trend_long['detail_line'] = gt_trend_long.apply(_detail_line, axis=1)

import os
OUT_DIR = "/home/h604827/ControlActions/RESULTS/gt_cause_pre_alarm_trend"
os.makedirs(OUT_DIR, exist_ok=True)
CSV_OUT = f"{OUT_DIR}/all_loops_gt_cause_pre_alarm_trend.csv"
gt_trend_long.to_csv(CSV_OUT, index=False)

print(f"Rows (loop x episode x cause tag): {len(gt_trend_long)}   "
      f"(alarm loops: {gt_trend_long['alarm_tag'].nunique()}, "
      f"distinct episodes: {gt_trend_long.groupby('alarm_tag')['cluster_id'].nunique().sum()})")
print("\nRows per alarm loop:\n" + gt_trend_long['alarm_tag'].value_counts().to_string())
print("\nDirection distribution:\n" + gt_trend_long['direction'].value_counts().to_string())
print("\nTrend-character distribution:\n" + gt_trend_long['trend_character'].value_counts().to_string())
print(f"\nSaved tidy table -> {CSV_OUT}")
_show = ['sheet', 'alarm_tag', 'excel_row', 'cluster_id', 'gt_token', 'pv_col', 'direction',
         'trend_character', 'net90_eu', 'pct_of_op_band', 'from_onset_min', 'coverage']
display(gt_trend_long[_show].head(25))


Rows (loop x episode x cause tag): 266   (alarm loops: 6, distinct episodes: 124)

Rows per alarm loop:
alarm_tag
1071    110
1016     58
1023     41
1104     22
1619     21
1009     14

Direction distribution:
direction
decreasing    136
increasing     98
stagnant       30
no_data         2

Trend-character distribution:
trend_character
strong_ramp    118
reversing       68
flat            28
late_move       28
weak_drift      20
no_data          2
oscillating      2

Saved tidy table -> /home/h604827/ControlActions/RESULTS/gt_cause_pre_alarm_trend/all_loops_gt_cause_pre_alarm_trend.csv


,sheet,alarm_tag,excel_row,cluster_id,gt_token,pv_col,direction,trend_character,net90_eu,pct_of_op_band,from_onset_min,coverage
0,1071 -Updated 2,1071,3,532,PIC_1013,03PIC_1013.PV,increasing,strong_ramp,43.055557,57.287447,90.0,1.0
1,1071 -Updated 2,1071,4,521,FI_1000,02FI_1000.PV,decreasing,strong_ramp,-0.719325,-295.005852,60.0,1.0
2,1071 -Updated 2,1071,5,517,PIC_1013,03PIC_1013.PV,stagnant,flat,0.032787,0.043625,NaN,1.0
3,1071 -Updated 2,1071,5,517,TIC_1092,03TIC_1092.PV,increasing,strong_ramp,3.182556,176.011255,89.0,1.0
4,1071 -Updated 2,1071,5,517,TI_1421,03TI_1421.PV,increasing,strong_ramp,3.200325,211.176557,90.0,1.0
5,1071 -Updated 2,1071,6,516,PIC_1013,03PIC_1013.PV,stagnant,flat,0.033300,0.044307,NaN,1.0
6,1071 -Updated 2,1071,6,516,FI_1000,02FI_1000.PV,decreasing,strong_ramp,-0.515265,-211.317665,74.0,1.0
7,1071 -Updated 2,1071,8,514,FI_1000,02FI_1000.PV,increasing,strong_ramp,0.132299,54.257707,90.0,1.0
8,1071 -Updated 2,1071,8,514,PIC_1013,03PIC_1013.PV,stagnant,flat,-0.016650,-0.022154,NaN,1.0
9,1071 -Updated 2,1071,9,493,FI_1000,02FI_1000.PV,decreasing,reversing,-0.177517,-72.802379,90.0,1.0


In [28]:
# ═══ Part 3.4 — write GT cause-tag behaviour INTO the cause cells (_kg workbook) ═
# Append the increasing/decreasing behaviour next to each ground-truth cause tag inside its
# Cause 1/2/3 cell (e.g. "PIC_1013 [increasing]"). The separate multi-line "GT pre-alarm trend"
# detail column is NOT added (removed on request) — the full per-tag detail is still in the
# Part 3.3 CSV (RESULTS/gt_cause_pre_alarm_trend/all_loops_gt_cause_pre_alarm_trend.csv).
import os
import shutil
import openpyxl

# behaviour per cause cell: (sheet, excel_row, cause_col) -> [(token, behaviour), ...]
beh_by_cell = {}
for (sheet, xr, col), g in gt_trend_long.groupby(['sheet', 'excel_row', 'cause_col']):
    beh_by_cell[(sheet, xr, col)] = [(rr['gt_token'], rr['behavior']) for _, rr in g.iterrows()]

# build on the Part-2 _kg workbook (KG + action columns already present); fall back to pristine copy
if not os.path.exists(SIT_OUTPUT):
    shutil.copy2(SIT_SOURCE, SIT_OUTPUT)
wb = openpyxl.load_workbook(SIT_OUTPUT)
# pristine cause text -> idempotent write-back regardless of re-runs
_pristine = {sheet: pd.read_excel(SIT_SOURCE, sheet_name=sheet, header=None) for sheet in SHEET_CFG}

for sheet in SHEET_CFG:
    ws = wb[sheet]
    raw = _pristine[sheet]
    n_written = 0
    for col in sheet_cause_cols.get(sheet, []):
        for xr in range(1, len(raw)):
            pieces = beh_by_cell.get((sheet, xr, col))
            if not pieces:
                continue
            orig = raw.iat[xr, col]
            orig = '' if (isinstance(orig, float) and pd.isna(orig)) else str(orig).strip()
            if len(pieces) == 1:
                new_val = f"{orig} [{pieces[0][1]}]"
            else:
                new_val = f"{orig} [" + "; ".join(f"{tok}: {beh}" for tok, beh in pieces) + "]"
            ws.cell(row=xr + 1, column=col + 1, value=new_val)   # openpyxl is 1-based
            n_written += 1
    print(f"  {sheet!r}: wrote behaviour into {n_written} cause cell(s)")

wb.save(SIT_OUTPUT)
print(f"\nSaved: {SIT_OUTPUT}")


  '1071 -Updated 2': wrote behaviour into 52 cause cell(s)
  'Updated LIC_1071_forSITVal': wrote behaviour into 57 cause cell(s)
  ' PIC_1104': wrote behaviour into 11 cause cell(s)
  'PIC1104-Upadated2': wrote behaviour into 11 cause cell(s)
  'TIC_1009-Updated ': wrote behaviour into 14 cause cell(s)
  'Updated LIC_1016': wrote behaviour into 58 cause cell(s)
  'Updated TIC_1023': wrote behaviour into 41 cause cell(s)
  'Updated LIC_1619': wrote behaviour into 21 cause cell(s)

Saved: /home/h604827/ControlActions/DATA/ADNOCGAS_Plant3Train1_RCA -After SIT Validation_with_control_actions_with_kg.xlsx


In [8]:
# ═══ Part 3.5 — visual validation: plot the 90-min pre-alarm window ════════════
import plotly.graph_objects as go

def _vmark(fig, xts, color, text, dash='dot'):
    """Datetime-safe vertical marker (add_vline chokes on Timestamp annotations)."""
    fig.add_shape(type='line', x0=xts, x1=xts, y0=0, y1=1, yref='paper',
                  line=dict(color=color, dash=dash, width=1.5))
    fig.add_annotation(x=xts, y=1.02, yref='paper', yanchor='bottom', text=text,
                       showarrow=False, font=dict(color=color, size=11))

def plot_gt_trend(i):
    """Plot raw + smoothed PV for gt_trend_long row i, with Theil-Sen line,
    dead-band, onset marker and operating limits."""
    r = gt_trend_long.iloc[i]
    anchor = pd.Timestamp(r['cstart']).floor('min')
    grid = pd.date_range(anchor - pd.Timedelta(minutes=PRE_MIN), anchor, freq='1min')
    s = pv_df[r['pv_col']].reindex(grid, method='nearest', tolerance=pd.Timedelta('30s'))
    s_s = (s.interpolate('linear', limit=SMOOTH_MIN, limit_direction='both')
             .rolling(SMOOTH_MIN, center=True, min_periods=2).median())

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=s, mode='lines', name='raw PV',
                             line=dict(color='lightgray', width=1)))
    fig.add_trace(go.Scatter(x=grid, y=s_s, mode='lines', name='smoothed (5-min median)',
                             line=dict(color='royalblue', width=2)))
    base = s_s.dropna()
    if len(base) and np.isfinite(r['slope90']):
        b0 = base.iloc[0]
        xm = (grid - grid[0]).total_seconds() / 60.0
        fig.add_trace(go.Scatter(x=grid, y=b0 + r['slope90'] * xm, mode='lines',
                                 name='Theil-Sen 90-min', line=dict(color='firebrick', dash='dash')))
        if np.isfinite(r['deadband_eu']):
            for sgn in (+1, -1):
                fig.add_trace(go.Scatter(x=grid, y=[b0 + sgn * r['deadband_eu']] * len(grid),
                                         mode='lines', line=dict(color='gray', dash='dot', width=1),
                                         name='dead-band', showlegend=(sgn == 1)))
    if r['pv_col'] in _ol.index:
        for lim, nm in [(_ol.at[r['pv_col'], 'LOWER_LIMIT'], 'low lim'),
                        (_ol.at[r['pv_col'], 'UPPER_LIMIT'], 'high lim')]:
            fig.add_hline(y=float(lim), line=dict(color='orange', dash='dot', width=1),
                          annotation_text=nm)
    if not pd.isna(r['onset_time']):
        _vmark(fig, pd.Timestamp(r['onset_time']), 'green', 'onset')
    _vmark(fig, anchor, 'black', 'alarm', dash='solid')
    fig.update_layout(
        title=(f"{r['sheet']} · row {r['excel_row']} · cluster {r['cluster_id']} · "
               f"{r['gt_token']} ({r['pv_col']})<br>"
               f"<sub>{_behavior_word(r)} [{r['trend_character']}] · Δ90={r['net90_eu']:+.2f} · "
               f"cov {r['coverage']:.0%}</sub>"),
        height=460, template='plotly_white', xaxis_title='time',
        yaxis_title=r['pv_col'], legend=dict(orientation='h', y=-0.2))
    return fig

# render one example per distinct trend character (skip no_data)
_seen, _n = set(), 0
for _i in range(len(gt_trend_long)):
    ch = gt_trend_long.iloc[_i]['trend_character']
    if ch in _seen or ch == 'no_data':
        continue
    _seen.add(ch); _n += 1
    plot_gt_trend(_i).show()
    if _n >= 6:
        break
print(f"Plotted {_n} example(s). Use plot_gt_trend(i) for any row i in gt_trend_long.")


Plotted 6 example(s). Use plot_gt_trend(i) for any row i in gt_trend_long.


---
# Part 4 — Per-Episode Control-Action Direction & Step Magnitude (filterable columns)

The *Action details* column is kept intact. After it we add, **per sheet**, machine-filterable columns
so the test team can slice operator actions by **direction** and **magnitude** and look for patterns
against the causal-tag behaviour.

For every SME episode, each **KG-kept control-action tag+type** operated in the alarm window (the same
`alarm − 30 min → alarm + 30 min` window as *Action details*) gets **two columns**:

- **`<tag>.<type> dir`** — `increased` / `decreased` / `no change`. If an operator moved the value both
  up and down within the episode, we keep the **majority** direction (by number of moves; the net-change
  sign breaks ties).
- **`<tag>.<type> avg|step|`** — the **average absolute step size** across that tag-type's moves
  (OP in %, SP in engineering units).

`OP` and `SP` of the same tag are **separate columns** (e.g. `03LIC_1071.OP` vs `03LIC_1071.SP`).
**MODE changes are ignored.** Columns are added **per sheet**, only for the tag-types that actually
appear in that loop's episodes (**most-operated first**), and appended after the existing columns.

**Colouring** — a cell is filled only when that tag-type was operated in the episode, so operated
tag-types stand out at a glance: **green = increased, red = decreased, grey = no change** (blank = not
operated).

_Requires Part 2 to have run (reuses `report`, `_tag_cache`, `kg_relevant_tags`, `WINDOW`, `SHEET_CFG`,
`SIT_OUTPUT`)._


In [18]:
# ═══ Part 4.1 — per-episode, per tag+type direction & avg step magnitude ═══════
from collections import Counter

ACT_TYPES = ['OP', 'SP']          # control-action types to summarise (MODE ignored)

def _to_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

def _tagtype_summary(win, kg_set):
    """{(source, type): (direction, avg_abs_step, n_moves)} for KG-kept OP/SP numeric moves."""
    df = win[win['Source'].isin(kg_set) & win['Description'].isin(ACT_TYPES)].copy()
    df['pv'] = df['PrevValue'].map(_to_float)
    df['vv'] = df['Value'].map(_to_float)
    df = df[df['pv'].notna() & df['vv'].notna()].sort_values('VT_Start')
    out = {}
    for (src, typ), grp in df.groupby(['Source', 'Description']):
        deltas = (grp['vv'] - grp['pv']).to_numpy()
        inc, dec = int((deltas > 0).sum()), int((deltas < 0).sum())
        net = float(grp['vv'].iloc[-1] - grp['pv'].iloc[0])
        if inc > dec:
            direction = 'increased'
        elif dec > inc:
            direction = 'decreased'
        else:                                   # tie -> net-change sign breaks it
            direction = ('increased' if net > 1e-9 else
                         'decreased' if net < -1e-9 else 'no change')
        out[(src, typ)] = (direction, round(float(np.abs(deltas).mean()), 2), len(deltas))
    return out

# per sheet: {excel_row: {(src,typ):(dir,avg,n)}} + tag-type column order (most-operated first)
sheet_actsummary, sheet_tagtypes = {}, {}
for sheet, (tag, *_rest) in SHEET_CFG.items():
    cb, catbl = _tag_cache[tag]
    kg_set = kg_relevant_tags(tag)
    rmap, freq = {}, Counter()
    sub = report[(report['sheet'] == sheet) & report['cluster_id'].notna()]
    for _, rr in sub.iterrows():
        cid, xr = int(rr['cluster_id']), int(rr['excel_row'])
        cs, ce = cb.at[cid, 'cstart'], cb.at[cid, 'cend']
        win = catbl[(catbl['cluster_id'] == cid) &
                    (catbl['VT_Start'] >= cs - WINDOW) & (catbl['VT_Start'] <= ce + WINDOW)]
        summ = _tagtype_summary(win, kg_set)
        if summ:
            rmap[xr] = summ
            freq.update(summ.keys())
    sheet_actsummary[sheet] = rmap
    sheet_tagtypes[sheet] = [k for k, _ in sorted(freq.items(), key=lambda kv: (-kv[1], kv[0]))]
    print(f"  {sheet!r}: {len(sheet_tagtypes[sheet])} tag-type columns "
          f"({2 * len(sheet_tagtypes[sheet])} excel cols), {len(rmap)} episodes with actions")

# quick look at the most-operated tag-types overall
_allfreq = Counter()
for sheet, rmap in sheet_actsummary.items():
    for summ in rmap.values():
        for k in summ:
            _allfreq[k] += 1
print("\nTop operated tag-types (episode count):")
for (src, typ), n in _allfreq.most_common(12):
    print(f"  {src}.{typ}: {n}")


  '1071 -Updated 2': 17 tag-type columns (34 excel cols), 19 episodes with actions
  'Updated LIC_1071_forSITVal': 17 tag-type columns (34 excel cols), 19 episodes with actions
  ' PIC_1104': 22 tag-type columns (44 excel cols), 7 episodes with actions
  'PIC1104-Upadated2': 22 tag-type columns (44 excel cols), 7 episodes with actions
  'TIC_1009-Updated ': 13 tag-type columns (26 excel cols), 7 episodes with actions
  'Updated LIC_1016': 13 tag-type columns (26 excel cols), 18 episodes with actions
  'Updated TIC_1023': 5 tag-type columns (10 excel cols), 9 episodes with actions
  'Updated LIC_1619': 7 tag-type columns (14 excel cols), 21 episodes with actions

Top operated tag-types (episode count):
  03PIC_1013.OP: 28
  03LIC_1034.SP: 20
  03FIC_3435.OP: 20
  03LIC_1619.OP: 20
  03HIC_1151.OP: 19
  03TIC_1009.SP: 13
  03LIC_1071.SP: 11
  03HIC_3132.OP: 11
  03LIC_1085.SP: 9
  03FIC_1258.OP: 8
  03PIC_3131.SP: 8
  03FIC_1085.OP: 7


In [29]:
# ═══ Part 4.2 — write direction + avg|step| columns (coloured) into the _kg wb ══
import openpyxl
from openpyxl.styles import PatternFill, Alignment, Font

DIR_FILL = {
    'increased': PatternFill('solid', fgColor='C6EFCE'),   # green
    'decreased': PatternFill('solid', fgColor='FFC7CE'),   # red
    'no change': PatternFill('solid', fgColor='D9D9D9'),   # grey
}
_HDR_ALIGN = Alignment(text_rotation=90, vertical='bottom', horizontal='center')

wb = openpyxl.load_workbook(SIT_OUTPUT)
for sheet in SHEET_CFG:
    ws = wb[sheet]
    tts = sheet_tagtypes[sheet]
    rmap = sheet_actsummary[sheet]
    colpos, c = {}, ws.max_column + 1
    for (src, typ) in tts:                      # two columns per tag-type
        label = f"{src}.{typ}"
        h1 = ws.cell(row=1, column=c,     value=f"{label} dir")
        h2 = ws.cell(row=1, column=c + 1, value=f"{label} avg|step|")
        for h in (h1, h2):
            h.font = Font(bold=True); h.alignment = _HDR_ALIGN
        ws.column_dimensions[h1.column_letter].width = 10
        ws.column_dimensions[h2.column_letter].width = 10
        colpos[(src, typ)] = (c, c + 1)
        c += 2
    for xr, summ in rmap.items():               # fill + colour operated cells
        for (src, typ), (direction, avg, n) in summ.items():
            dcol, mcol = colpos[(src, typ)]
            dc = ws.cell(row=xr + 1, column=dcol, value=direction)
            mc = ws.cell(row=xr + 1, column=mcol, value=avg)
            fill = DIR_FILL.get(direction)
            if fill:
                dc.fill = fill; mc.fill = fill
    if tts:
        ws.row_dimensions[1].height = 130
    print(f"  {sheet!r}: wrote {len(tts)} tag-type pairs ({2 * len(tts)} cols), coloured operated cells")

wb.save(SIT_OUTPUT)
print(f"\nSaved: {SIT_OUTPUT}")


  '1071 -Updated 2': wrote 17 tag-type pairs (34 cols), coloured operated cells
  'Updated LIC_1071_forSITVal': wrote 17 tag-type pairs (34 cols), coloured operated cells
  ' PIC_1104': wrote 22 tag-type pairs (44 cols), coloured operated cells
  'PIC1104-Upadated2': wrote 22 tag-type pairs (44 cols), coloured operated cells
  'TIC_1009-Updated ': wrote 13 tag-type pairs (26 cols), coloured operated cells
  'Updated LIC_1016': wrote 13 tag-type pairs (26 cols), coloured operated cells
  'Updated TIC_1023': wrote 5 tag-type pairs (10 cols), coloured operated cells
  'Updated LIC_1619': wrote 7 tag-type pairs (14 cols), coloured operated cells

Saved: /home/h604827/ControlActions/DATA/ADNOCGAS_Plant3Train1_RCA -After SIT Validation_with_control_actions_with_kg.xlsx
